# Build Pinecone Index — Multi-Seed CLIP Fused Embeddings

Creates **frozen** namespaces once (seed-independent), then for **each seed**
builds finetuned namespaces using that seed's `clip_best_seed{S}.pt`.

| Namespace | Condition | CLIP weights | α | Seed |
|---|---|---|---|---|
| `frozen-alpha-1.0` | A | pretrained (frozen) | 1.0 | — |
| `frozen-alpha-0.7` | B | pretrained (frozen) | 0.7 | — |
| `frozen-alpha-0.5` | B | pretrained (frozen) | 0.5 | — |
| `finetuned-alpha-0.7-seed{S}` | C | clip_best_seed{S}.pt | 0.7 | per-seed |
| `finetuned-alpha-0.5-seed{S}` | C | clip_best_seed{S}.pt | 0.5 | per-seed |


In [ ]:
!pip install -q pinecone open_clip_torch

In [ ]:
# ==============================================================================
# CONFIG — fill in before running
# ==============================================================================
PINECONE_API_KEY  = "YOUR_PINECONE_API_KEY"
INDEX_NAME        = "vr-clothing-gallery"

DATASET_ROOT      = "/kaggle/input/datasets/vinay1706/deepfashion-cropped"
GALLERY_CSV       = f"{DATASET_ROOT}/gallery.csv"
CAPTIONS_CSV      = f"{DATASET_ROOT}/blip2_captions_gallery.csv"
IMAGE_ROOT        = DATASET_ROOT

CLIP_MODEL_NAME   = "ViT-L-14"
CLIP_PRETRAINED   = "openai"
EMBED_DIM         = 768
BATCH_SIZE        = 128
UPSERT_BATCH      = 100

# ── Multi-seed configuration ──────────────────────────────────────────────────
# List all seeds you have trained checkpoints for.
# Each seed's checkpoint should be at: {DATASET_ROOT}/clip_best_seed{S}.pt
SEEDS = [42, 123]

# Frozen namespaces (built once, seed-independent)
FROZEN_NAMESPACES = [
    ("frozen-alpha-1.0", 1.0),
    ("frozen-alpha-0.7", 0.7),
    ("frozen-alpha-0.5", 0.5),
]

# Finetuned alpha values (namespaces are created per-seed as finetuned-alpha-{A}-seed{S})
FINETUNED_ALPHAS = [0.7, 0.5]


In [ ]:
import os
import torch
import torch.nn.functional as F
import open_clip
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from pinecone import Pinecone, ServerlessSpec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# ── Load CSVs ─────────────────────────────────────────────────────────────────
gallery_df  = pd.read_csv(GALLERY_CSV)
captions_df = pd.read_csv(CAPTIONS_CSV)

df = gallery_df.merge(captions_df[["image_name", "blip2_caption"]], on="image_name", how="left")
df["blip2_caption"] = df["blip2_caption"].fillna("")

print(f"Gallery rows : {len(df)}")
print(f"Unique items : {df['item_id'].nunique()}")
df.head(3)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class GalleryDataset(Dataset):
    def __init__(self, df, image_root, preprocess):
        self.df         = df.reset_index(drop=True)
        self.image_root = image_root
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_root, row["image_name"])
        try:
            image = self.preprocess(Image.open(img_path).convert("RGB"))
        except Exception:
            image = torch.zeros(3, 224, 224)
        return image, row["blip2_caption"], row["image_name"], str(row["item_id"])

In [ ]:
# ── Helper: load CLIP (frozen or fine-tuned) ──────────────────────────────────
def load_clip(checkpoint_path=None):
    \"\"\"Load CLIP model. If checkpoint_path is given, load fine-tuned weights.\"\"\"
    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED
    )
    if checkpoint_path is not None:
        ckpt       = torch.load(checkpoint_path, map_location=device)
        state_dict = ckpt.get("model_state", ckpt)
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=False)
        print(f"  Loaded fine-tuned weights from {checkpoint_path}")
    else:
        print("  Using frozen pretrained weights")
    return model.to(device).eval(), preprocess, open_clip.get_tokenizer(CLIP_MODEL_NAME)


# ── Helper: embed entire gallery ──────────────────────────────────────────────
@torch.no_grad()
def embed_gallery(model, preprocess, tokenizer):
    dataset = GalleryDataset(df, IMAGE_ROOT, preprocess)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         num_workers=4, pin_memory=True, shuffle=False)
    vis_list, txt_list, names_list, ids_list = [], [], [], []

    with torch.amp.autocast('cuda'):
        for images, captions, image_names, item_ids in tqdm(loader, desc="  Embedding"):
            images = images.to(device)
            tokens = tokenizer(list(captions)).to(device)
            vis_emb = F.normalize(model.encode_image(images).float(), dim=-1)
            txt_emb = F.normalize(model.encode_text(tokens).float(),  dim=-1)
            vis_list.append(vis_emb.cpu())
            txt_list.append(txt_emb.cpu())
            names_list.extend(image_names)
            ids_list.extend(item_ids)

    return torch.cat(vis_list), torch.cat(txt_list), names_list, ids_list


# ── Helper: upsert one namespace ──────────────────────────────────────────────
def upsert_namespace(namespace, alpha, vis_embs, txt_embs, image_names, item_ids):
    fused = F.normalize(alpha * vis_embs + (1 - alpha) * txt_embs, dim=-1).numpy()
    vectors = [
        {
            "id"      : name.replace("/", "|"),
            "values"  : vec.tolist(),
            "metadata": {"item_id": item_id, "image_name": name, "alpha": alpha}
        }
        for vec, name, item_id in zip(fused, image_names, item_ids)
    ]
    for start in tqdm(range(0, len(vectors), UPSERT_BATCH),
                      desc=f"  Upserting {namespace}"):
        index.upsert(vectors=vectors[start:start + UPSERT_BATCH], namespace=namespace)
    print(f"  ✓ {len(vectors)} vectors → '{namespace}'")

In [ ]:
# ── Init Pinecone index ───────────────────────────────────────────────────────
pc = Pinecone(api_key=PINECONE_API_KEY)

if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name      = INDEX_NAME,
        dimension = EMBED_DIM,
        metric    = "cosine",
        spec      = ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{INDEX_NAME}' created.")
else:
    print(f"Index '{INDEX_NAME}' already exists — will add/overwrite namespaces.")

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

In [ ]:
# ==============================================================================
# STEP 1: Build FROZEN namespaces (seed-independent, only needs to run once)
# ==============================================================================
print("=" * 60)
print("Loading frozen pretrained CLIP...")
print("=" * 60)

frozen_model, frozen_preprocess, frozen_tokenizer = load_clip(checkpoint_path=None)
frozen_vis, frozen_txt, image_names, item_ids = embed_gallery(
    frozen_model, frozen_preprocess, frozen_tokenizer
)
print(f"  Embedded {len(image_names)} images with frozen CLIP.")

for namespace, alpha in FROZEN_NAMESPACES:
    print(f"\nBuilding namespace '{namespace}' (α={alpha})...")
    upsert_namespace(namespace, alpha, frozen_vis, frozen_txt, image_names, item_ids)

del frozen_model, frozen_vis, frozen_txt
torch.cuda.empty_cache()
print("\n✓ Frozen namespaces complete.")

In [ ]:
# ==============================================================================
# STEP 2: Build FINETUNED namespaces (one set per seed)
# ==============================================================================
for seed in SEEDS:
    checkpoint_path = f"{DATASET_ROOT}/clip_best_seed{seed}.pt"
    
    print(f"\n{'=' * 60}")
    print(f"SEED {seed} — Loading fine-tuned CLIP from {checkpoint_path}")
    print("=" * 60)
    
    if not os.path.exists(checkpoint_path):
        print(f"  ⚠ Checkpoint not found: {checkpoint_path} — SKIPPING seed {seed}")
        continue
    
    ft_model, ft_preprocess, ft_tokenizer = load_clip(checkpoint_path=checkpoint_path)
    ft_vis, ft_txt, image_names, item_ids = embed_gallery(
        ft_model, ft_preprocess, ft_tokenizer
    )
    print(f"  Embedded {len(image_names)} images with seed-{seed} finetuned CLIP.")
    
    for alpha in FINETUNED_ALPHAS:
        namespace = f"finetuned-alpha-{alpha}-seed{seed}"
        print(f"\nBuilding namespace '{namespace}' (α={alpha})...")
        upsert_namespace(namespace, alpha, ft_vis, ft_txt, image_names, item_ids)
    
    del ft_model, ft_vis, ft_txt
    torch.cuda.empty_cache()
    print(f"\n✓ Seed {seed} finetuned namespaces complete.")

print("\n" + "=" * 60)
print("All namespaces built.")
print(index.describe_index_stats())

In [ ]:
# ── Smoke test — top-3 from each namespace ────────────────────────────────────
# Use frozen model to generate a test query vector
test_model, test_prep, _ = load_clip(checkpoint_path=None)
test_img_path = os.path.join(IMAGE_ROOT, df.iloc[0]["image_name"])
test_img = test_prep(Image.open(test_img_path).convert("RGB")).unsqueeze(0).to(device)
with torch.no_grad():
    test_vec = F.normalize(test_model.encode_image(test_img).float(), dim=-1)
test_vec = test_vec.cpu().numpy().tolist()[0]
del test_model
torch.cuda.empty_cache()

# Collect all namespace names
all_namespaces = [ns for ns, _ in FROZEN_NAMESPACES]
for seed in SEEDS:
    for alpha in FINETUNED_ALPHAS:
        all_namespaces.append(f"finetuned-alpha-{alpha}-seed{seed}")

print("Smoke test — querying each namespace with gallery image 0:")
for namespace in all_namespaces:
    result = index.query(vector=test_vec, top_k=3,
                         namespace=namespace, include_metadata=True)
    print(f"\n  [{namespace}]")
    for m in result["matches"]:
        print(f"    score={m['score']:.4f}  item={m['metadata']['item_id']}  img={m['metadata']['image_name']}")